<a href="https://colab.research.google.com/github/JariusF10-8/bt4222/blob/main/Dataset_Overview.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BT4222 — MIND-small: Data Loading and Overview

Run each code cell from top to bottom in Google Colab. This starter uses your official **MINDsmall_train** files. It reads the files and reports basic checks; it does not modify the files in Drive.


| Folder | Role |
|---|---|
| MINDsmall_train | Training split; used for the initial inspection here |
| MINDsmall_dev | Development/validation split; keep separate for later evaluation |
| MINDlarge_train / MINDlarge_dev / MINDlarge_test | Large dataset; not needed in this starter |

| File | Contents | Load now? |
|---|---|---|
| behaviors.tsv | Impression events, user IDs, time, previous click history, and displayed candidates with click labels | Yes |
| news.tsv | Article IDs, categories, titles, abstracts, URLs, and entity annotations | Yes |
| entity_embedding.vec | Knowledge-graph entity vectors | Later, for entity features |
| relation_embedding.vec | Knowledge-graph relation vectors | Later, if the modelling approach uses relations |


## 1. Imports
Pandas loads the files into DataFrames (tables). Path builds file locations. The csv setting used below preserves literal quotation marks in titles and entity JSON.


In [ ]:
from pathlib import Path
import csv
import sys

import pandas as pd
from IPython.display import display

print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)


Python version: 3.13.15
Pandas version: 2.2.3


## 2. Mount Google Drive
Run this in Google Colab and follow the Google sign-in prompt. Mounting makes Drive accessible through `/content/drive`. It does not copy your dataset into GitHub.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Set the data location and check the files
Google Drive's **My Drive** appears as **MyDrive** in the mounted path. Spaces and capitalisation in the remaining folder names must match.

This cell stops with a clear message if the folder or a required file is missing. It does not create an empty folder that could hide a path mistake.


In [ ]:
DATA_ROOT = Path("/content/drive/MyDrive/BT4222 Group Project/Microsoft MIND")
SPLIT = "MINDsmall_train"

DATA_DIR = DATA_ROOT / SPLIT
BEHAVIORS_FILE = DATA_DIR / "behaviors.tsv"
NEWS_FILE = DATA_DIR / "news.tsv"

if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        f"Folder not found: {DATA_DIR}\n"
        "Check that Drive is mounted with the correct Google account, "
        "then check folder names and any extra folder created when extracting the ZIP."
    )

for file_path in [BEHAVIORS_FILE, NEWS_FILE]:
    if not file_path.is_file():
        raise FileNotFoundError(
            f"Required file not found: {file_path}\n"
            "The extracted TSV files should be directly inside the selected split folder."
        )
    print(f"Found: {file_path.name} ({file_path.stat().st_size / 1024**2:.1f} MiB)")

print("Selected split:", SPLIT)
print("Data folder:", DATA_DIR)


Found: behaviors.tsv (87.8 MiB)
Found: news.tsv (39.3 MiB)
Selected split: MINDsmall_train
Data folder: /content/drive/MyDrive/BT4222 Group Project/Microsoft MIND/MINDsmall_train


## 4. Load the two TSV files
- `sep="\t"`: fields are separated by tabs.
- `header=None`: the files do not contain a header row.
- `names=...`: provide the column names.
- `dtype="string"`: keep identifiers and other raw fields as strings.
- `keep_default_na=False, na_values=[""]`: treat empty fields as missing while preserving literal text such as "NA".
- `quoting=csv.QUOTE_NONE`: read quotation marks literally in these tab-separated records.

Histories, impressions and entity JSON remain in their original string form for now. No rows are dropped and IDs are not remapped.


In [ ]:
BEHAVIORS_COLUMNS = [
    "impression_id", "user_id", "time", "history", "impressions"
]

NEWS_COLUMNS = [
    "news_id", "category", "subcategory", "title", "abstract",
    "url", "title_entities", "abstract_entities"
]

behaviors = pd.read_csv(
    BEHAVIORS_FILE,
    sep="\t",
    header=None,
    names=BEHAVIORS_COLUMNS,
    dtype="string",
    keep_default_na=False,
    na_values=[""],
    quoting=csv.QUOTE_NONE,
    encoding="utf-8",
)

news = pd.read_csv(
    NEWS_FILE,
    sep="\t",
    header=None,
    names=NEWS_COLUMNS,
    dtype="string",
    keep_default_na=False,
    na_values=[""],
    quoting=csv.QUOTE_NONE,
    encoding="utf-8",
)

print("behaviors shape (rows, columns):", behaviors.shape)
print("news shape (rows, columns):", news.shape)


behaviors shape (rows, columns): (156965, 5)
news shape (rows, columns): (51282, 8)


## 5. Preview rows and column types
One behaviors row represents one impression event, which can contain multiple candidate articles. One news row describes an article. The preview is just a small sample, not a full analysis.


In [ ]:
print("First 5 behavior records:")
display(behaviors.head())

print("First 5 article records:")
display(news.head())

print("Behavior column types:")
display(behaviors.dtypes.rename("data_type").to_frame())

print("News column types:")
display(news.dtypes.rename("data_type").to_frame())


First 5 behavior records:


,impression_id,user_id,time,history,impressions
0,1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689-1 N35729-0
1,2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...
2,3,U73700,11/14/2019 7:01:48 AM,N10732 N25792 N7563 N21087 N41087 N5445 N60384...,N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N...
3,4,U34670,11/11/2019 5:28:05 AM,N45729 N2203 N871 N53880 N41375 N43142 N33013 ...,N35729-0 N33632-0 N49685-1 N27581-0
4,5,U8125,11/12/2019 4:11:21 PM,N10078 N56514 N14904 N33740,N39985-0 N36050-0 N16096-0 N8400-1 N22407-0 N6...


First 5 article records:


,news_id,category,subcategory,title,abstract,url,title_entities,abstract_entities
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,[],"[{""Label"": ""National Basketball Association"", ..."
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...",https://assets.msn.com/labs/mind/AAAKEkt.html,"[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI...","[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI..."


Behavior column types:


,data_type
impression_id,string[python]
user_id,string[python]
time,string[python]
history,string[python]
impressions,string[python]


News column types:


,data_type
news_id,string[python]
category,string[python]
subcategory,string[python]
title,string[python]
abstract,string[python]
url,string[python]
title_entities,string[python]
abstract_entities,string[python]


## 6. Inspect basic counts
Counts apply only to the selected split. Distinct articles in news.tsv are metadata coverage, not the number of candidate exposures.

Blank or whitespace-only histories are counted as unavailable supplied histories. The number of affected impression rows differs from the number of distinct affected users. A user may have missing history in one record and a nonempty history in another.

Duplicate-row counts compare all original columns. Duplicate IDs are reported separately; review any such findings before deleting records.


In [ ]:
missing_history = behaviors["history"].fillna("").str.strip().eq("")

basic_counts = pd.Series({
    "Distinct users in behavior records": behaviors["user_id"].nunique(),
    "Impression rows": len(behaviors),
    "Distinct impression IDs": behaviors["impression_id"].nunique(),
    "Article metadata rows": len(news),
    "Distinct article IDs in metadata": news["news_id"].nunique(),
    "Impression rows with unavailable history": int(missing_history.sum()),
    "Distinct users with at least one unavailable history": (
        behaviors.loc[missing_history, "user_id"].nunique()
    ),
    "Fully duplicated behavior rows": int(behaviors.duplicated().sum()),
    "Fully duplicated news rows": int(news.duplicated().sum()),
    "Repeated nonmissing impression IDs (beyond first)": int(
        behaviors["impression_id"].dropna().duplicated().sum()
    ),
    "Repeated nonmissing article IDs (beyond first)": int(
        news["news_id"].dropna().duplicated().sum()
    ),
}, name="count").to_frame()

display(basic_counts)


,count
Distinct users in behavior records,50000
Impression rows,156965
Distinct impression IDs,156965
Article metadata rows,51282
Distinct article IDs in metadata,51282
Impression rows with unavailable history,3238
Distinct users with at least one unavailable history,892
Fully duplicated behavior rows,0
Fully duplicated news rows,0
Repeated nonmissing impression IDs (beyond first),0


## 7. Inspect missingness
Report missing/blank counts, percentages and distinct nonmissing values for each raw field. A literal `[]` in an entity column is an empty annotation list, not a missing string; it is not counted as missing here.


In [ ]:
def field_overview(df):
    # Identify missing, empty, and whitespace-only strings without changing df.
    blank = df.apply(lambda column: column.fillna("").str.strip().eq(""))
    nonblank = df.mask(blank)

    result = pd.DataFrame({
        "missing_or_blank_count": blank.sum(),
        "distinct_nonmissing_values": nonblank.nunique(),
    })
    if len(df):
        result["missing_or_blank_pct"] = (blank.mean() * 100).round(2)
    else:
        result["missing_or_blank_pct"] = float("nan")

    return result[[
        "missing_or_blank_count",
        "missing_or_blank_pct",
        "distinct_nonmissing_values",
    ]]

print("Behavior field overview:")
display(field_overview(behaviors))

print("News field overview:")
display(field_overview(news))


Behavior field overview:


,missing_or_blank_count,missing_or_blank_pct,distinct_nonmissing_values
impression_id,0,0.0,156965
user_id,0,0.0,50000
time,0,0.0,125501
history,3238,2.06,48868
impressions,0,0.0,143617


News field overview:


,missing_or_blank_count,missing_or_blank_pct,distinct_nonmissing_values
news_id,0,0.0,51282
category,0,0.0,17
subcategory,0,0.0,264
title,0,0.0,50434
abstract,2666,5.2,47309
url,0,0.0,51282
title_entities,0,0.0,34475
abstract_entities,0,0.0,36279


## 8. Check the timestamp range
The documented format is month/day/year followed by a 12-hour time and AM/PM. Parse into a separate Series so the original time column is retained. Unparseable nonblank strings are reported instead of silently disappearing.

These are recorded dataset times. No local timezone is assumed.


In [ ]:
impression_times = pd.to_datetime(
    behaviors["time"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce",
)

time_present = behaviors["time"].fillna("").str.strip().ne("")
invalid_time = time_present & impression_times.isna()

print("Earliest parsed impression time:", impression_times.min())
print("Latest parsed impression time:", impression_times.max())
print("Missing/blank timestamps:", int((~time_present).sum()))
print("Nonblank timestamps that could not be parsed:", int(invalid_time.sum()))

if invalid_time.any():
    display(behaviors.loc[invalid_time, ["impression_id", "time"]].head())


Earliest parsed impression time: 2019-11-09 00:00:19
Latest parsed impression time: 2019-11-14 23:59:13
Missing/blank timestamps: 0
Nonblank timestamps that could not be parsed: 0


## What comes next?
Once these cells run successfully, use `behaviors` and `news` for your group's EDA.

- Parse history strings into article ID lists and calculate history lengths.
- Parse candidate strings into article IDs and 0/1 labels while preserving impression IDs.
- Calculate slate lengths, exposures, clicks and click rates.
- Inspect text and entity information.
- Keep the development split separate. If you later inspect it, use separately named DataFrames or a separate notebook; avoid accidentally treating development records as training data.

The outputs are intentionally blank in this delivered notebook because your actual TSV files were not provided here. Run it in your Colab account to obtain your own results.

## References
- [Official MIND dataset website](https://msnews.github.io/)
- [Official MIND file format](https://github.com/msnews/msnews.github.io/blob/master/assets/doc/introduction.md)
